# Classification: hyperparameter tuning and cross-validation

Companion to `model_classification_dev.ipynb`. It uses the same features (`NUM_COLS + CAT_COLS`), the same as-of-run-date labels, the same run date (2018-06-02) and the same five classifiers. It adds:

1. **A chronological hold-out.** The newest 30 days of the training window are the test set. This replaces the random stratified split.
2. **Time-series cross-validation** on the training part. Each fold trains on earlier approval days and validates on later ones.
3. **Hyperparameter tuning** with `RandomizedSearchCV` on those folds, optimising **PR-AUC** (average precision). PR-AUC is the brief's recommended metric for an imbalanced target (~13% late).
4. **Threshold selection.** The late/on-time cut-off is chosen from out-of-fold CV scores (max F1) instead of a fixed 0.5.
5. **Default vs tuned, compared on:**
   - the hold-out;
   - a pooled monthly backtest (April–July 2018) that follows the operational flow of `model_classification_dev.ipynb`.

The tuned parameters and thresholds are saved to `results/classification_tuning.json`, so `model_classification_dev.ipynb` can load them.

# Updates
- 20261001 - Initial tuning/CV notebook

# Environment set-up

In [ ]:
import json, os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

from src import RANDOM_STATE, LOOKBACK, PXD
from src.data import load_olist_data
from src.features import build_feature_table, NUM_COLS, CAT_COLS, LEAKAGE_COLS
from src.labels import get_required_dates, labels_as_of, attach_prediction_labels
from src.splits import chronological_split, day_blocked_time_series_folds
from src.evaluation import classification_metrics
from src.tuning import (build_classifiers, cv_summary, tune_classifier, out_of_fold_scores,
                        best_f1_threshold, params_to_json, CLASSIFIER_SEARCH_SPACES, MODEL_FAMILY)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

# Constants

In [ ]:
RUN_DATE = '2018-06-02'   # Same run date as model_classification_dev.ipynb
TEST_DAYS = 30            # Newest approval days held out (~1 month, as in the dev notebook)
N_CV_FOLDS = 5
# Random-search iterations per model; kept small for laptop run times (~10 min on 4 cores)
N_ITER = {'Logistic Regression': 12, 'Decision Tree': 15, 'Random Forest': 8,
          'XGBoost': 12, 'LightGBM': 12}
FEATURE_COLS = NUM_COLS + CAT_COLS
LABEL_COL = 'dpt_var_late_delivery'
assert not set(FEATURE_COLS) & set(LEAKAGE_COLS), 'Outcome columns used as features'

# Load data and training window

In [ ]:
olist = load_olist_data(verbose=False)
orders = olist['orders']
final_df = build_feature_table(olist)

start_date, end_date, inference_date = get_required_dates(RUN_DATE, lookback=LOOKBACK, pXd=PXD)
window = labels_as_of(final_df[final_df['order_approved_dt'].between(start_date, end_date)], RUN_DATE)
labelled = window[window['label_as_of_run'].notna()].copy()
labelled[LABEL_COL] = labelled['label_as_of_run'].astype(int)
print(f'Known labels: {len(labelled):,}/{len(window):,}; late rate {labelled[LABEL_COL].mean():.2%}')
assert len(labelled) == 39_956, 'Should match model_classification_dev.ipynb'

# Chronological split and CV folds
- **Test:** the last 30 approval days of the window.
- **Train:** everything before that.
- **CV folds:** cut by approval day on the training part only, with an expanding window. Fold *k* trains on blocks 1…*k* and validates on block *k*+1. No approval day is split across train and validation.

**Why not a random stratified split?** The dev notebook uses one, so its test set is spread across the whole window and the model trains on orders *after* some of the orders it is tested on. In operation the model only ever scores *future* orders. Time-ordered splits reproduce that, so their scores are what we can honestly expect.

Each part's late rate is shown below, since we don't stratify. The comparison further down measures how much the random split flatters the scores.

In [ ]:
train_df, test_df = chronological_split(labelled, date_col='order_approved_dt', test_days=TEST_DAYS)
train_df = train_df.sort_values('order_approved_dt', kind='stable')
X_train, y_train = train_df[FEATURE_COLS], train_df[LABEL_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[LABEL_COL]
print(f'Train: {train_df["order_approved_dt"].min():%Y-%m-%d} to {train_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(train_df):,} orders, {y_train.mean():.2%} late)')
print(f'Test:  {test_df["order_approved_dt"].min():%Y-%m-%d} to {test_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(test_df):,} orders, {y_test.mean():.2%} late)')

cv_folds = day_blocked_time_series_folds(train_df['order_approved_dt'], n_splits=N_CV_FOLDS)
display(pd.DataFrame([{
    'fold': k, 'train_orders': len(tr), 'valid_orders': len(va),
    'valid_from': train_df['order_approved_dt'].iloc[va].min().date(),
    'valid_to': train_df['order_approved_dt'].iloc[va].max().date(),
    'valid_late_rate': y_train.iloc[va].mean(),
} for k, (tr, va) in enumerate(cv_folds, 1)]).set_index('fold'))

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
default_models = build_classifiers(NUM_COLS, CAT_COLS, scale_pos_weight)

# Cross-validated defaults
These are the untuned settings from `model_classification_dev.ipynb`, scored on the time-series folds:
- **PR-AUC**: the primary metric. A random model scores the late rate, about 0.13.
- **ROC-AUC**.
- **F1** at the default 0.5 threshold.

In [ ]:
cv_rows = []
for name, model in default_models.items():
    cv_rows.append({'model': name, 'variant': 'default', 'family': MODEL_FAMILY[name],
                    **cv_summary(model, X_train, y_train, cv_folds)})
display(pd.DataFrame(cv_rows).set_index(['model', 'variant']))

## How much does a random split flatter the scores?
This scores default LightGBM with stratified *random* 5-fold CV on the same training rows. The difference from the time-series CV above is the optimism we would report by shuffling future orders into training.

In [ ]:
random_cv = cv_summary(default_models['LightGBM'], X_train, y_train,
                       StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
time_cv = next(r for r in cv_rows if r['model'] == 'LightGBM')
display(pd.DataFrame({'random stratified 5-fold': random_cv, 'time-series 5-fold': time_cv})
        .loc[['cv_pr_auc_mean', 'cv_roc_auc_mean', 'cv_f1_at_0.5_mean']])

# Hyperparameter tuning
Each model runs a random search over a small grid (`src/tuning.py`, `CLASSIFIER_SEARCH_SPACES`):
- It uses the same time-series folds and maximises mean PR-AUC.
- The class-imbalance weighting stays as in the dev notebook: `class_weight='balanced'`, or `scale_pos_weight` for the boosting models.
- The test set is never used for selection.

In [ ]:
tuned_models, best_params = {}, {}
for name, model in default_models.items():
    tuned, params, search = tune_classifier(model, CLASSIFIER_SEARCH_SPACES[name], X_train, y_train,
                                            cv_folds, n_iter=N_ITER[name])
    tuned_models[name], best_params[name] = tuned, params
    cv_rows.append({'model': name, 'variant': 'tuned', 'family': MODEL_FAMILY[name],
                    **cv_summary(tuned, X_train, y_train, cv_folds)})
    print(f'{name}: {params_to_json(params)}')

cv_results = pd.DataFrame(cv_rows).set_index(['model', 'variant']).sort_index()
display(cv_results)
gain = (cv_results.xs('tuned', level='variant')['cv_pr_auc_mean']
        - cv_results.xs('default', level='variant')['cv_pr_auc_mean'])
display(gain.rename('cv_pr_auc_gain_from_tuning').to_frame())

# Threshold selection
At 0.5 the balanced models flag far more orders than are actually late, so precision is low.
- For each tuned model, this picks the threshold that maximises F1 on the **out-of-fold** CV scores.
- Each score comes from a model that never saw that order, so choosing the threshold doesn't leak information.
- Operations may instead prefer a capacity rule (e.g. contact the top 10% riskiest orders each day). The decile coverage tables in the dev notebook support that.

In [ ]:
thresholds = {}
for name, model in tuned_models.items():
    oof = out_of_fold_scores(model, X_train, y_train, cv_folds)
    thresholds[name], oof_f1 = best_f1_threshold(y_train, oof)
    print(f'{name}: threshold {thresholds[name]:.3f} (out-of-fold F1 {oof_f1:.3f})')

# Hold-out evaluation (newest 30 days of the window)

In [ ]:
holdout_rows = []
for variant, model_set in [('default', default_models), ('tuned', tuned_models)]:
    for name, model in model_set.items():
        fitted = clone(model).fit(X_train, y_train)
        prob = fitted.predict_proba(X_test)[:, list(fitted.classes_).index(1)]
        cutoffs = [('0.5', 0.5)] + ([('CV-tuned', thresholds[name])] if variant == 'tuned' else [])
        for label, cutoff in cutoffs:
            metrics = classification_metrics(y_test, (prob >= cutoff).astype(int), prob)
            holdout_rows.append({'model': name, 'variant': variant, 'threshold': label,
                                 'flagged_pct': (prob >= cutoff).mean(), **metrics})
holdout_results = pd.DataFrame(holdout_rows).set_index(['model', 'variant', 'threshold'])
display(holdout_results[['pr_auc', 'roc_auc', 'precision', 'recall', 'f1_score',
                         'balanced_accuracy', 'flagged_pct']])

# Pooled monthly backtest (operational flow)
This mirrors the backtest in `model_classification_dev.ipynb`:
1. For each month, fit on the 2nd using the full labelled window as of that date.
2. Score every order approved that month that wasn't delivered by the end of its approval day.
3. Attach each order's actual label at approval day + 45 days.

Default and tuned models are both compared, with the tuned ones at their CV threshold.

**Months: April–July 2018.** The hyperparameters and thresholds were chosen using approvals up to 2018-03-18, so these months were never used for tuning. Earlier months would reuse the tuning data and look better than they should.

In [ ]:
BACKTEST_RUN_DATES = ['2018-04-02', '2018-05-02', '2018-06-02', '2018-07-02']

def backtest_month(run_date):
    start, end, _ = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD, verbose=False)
    history = labels_as_of(final_df[final_df['order_approved_dt'].between(start, end)], run_date)
    history = history[history['label_as_of_run'].notna()]
    y_fit = history['label_as_of_run'].astype(int)
    month = pd.Period(run_date, 'M')
    cohort = final_df[final_df['order_approved_dt'].dt.to_period('M').eq(month)]
    delivered_day = cohort['order_delivered_customer_date'].dt.normalize()
    cohort = cohort[delivered_day.isna() | delivered_day.gt(cohort['order_approved_dt'])]

    ratio = (y_fit == 0).sum() / (y_fit == 1).sum()
    defaults = build_classifiers(NUM_COLS, CAT_COLS, ratio)
    frames = []
    for variant in ['default', 'tuned']:
        for name, model in defaults.items():
            model = clone(model).set_params(**best_params[name]) if variant == 'tuned' else clone(model)
            model.fit(history[FEATURE_COLS], y_fit)
            frame = cohort[['order_id', 'order_approved_dt']].copy()
            frame['predicted_probability'] = model.predict_proba(cohort[FEATURE_COLS])[:, 1]
            frame['model'], frame['variant'], frame['month'] = name, variant, str(month)
            frames.append(frame)
    return pd.concat(frames, ignore_index=True)

backtest = pd.concat(Parallel(n_jobs=-1)(delayed(backtest_month)(d) for d in BACKTEST_RUN_DATES),
                     ignore_index=True)
labels = attach_prediction_labels(backtest[['order_id']].drop_duplicates()
                                  .assign(predicted_probability=0.0), orders, lookback=LOOKBACK)
backtest = backtest.merge(labels[['order_id', 'actual_label', 'label_status']], on='order_id')
print(f'{backtest["order_id"].nunique():,} scored orders; '
      f'known labels {backtest.drop_duplicates("order_id")["actual_label"].notna().mean():.2%}')

In [ ]:
def backtest_metrics(group):
    known = group[group['actual_label'].notna()]
    y = known['actual_label'].astype(int)
    name, variant = group.name[-2], group.name[-1]
    cutoff = thresholds[name] if variant == 'tuned' else 0.5
    return pd.Series(classification_metrics(y, (known['predicted_probability'] >= cutoff).astype(int),
                                            known['predicted_probability']))

monthly_metrics = (backtest.groupby(['month', 'model', 'variant'])
                   .apply(backtest_metrics).reset_index())
display(monthly_metrics.pivot_table(index=['model', 'variant'], columns='month', values='pr_auc'))

summary = monthly_metrics.groupby(['model', 'variant'])[
    ['pr_auc', 'roc_auc', 'precision', 'recall', 'f1_score']].agg(['mean', 'std'])
pooled = (backtest.groupby(['model', 'variant']).apply(backtest_metrics)
          [['pr_auc', 'precision', 'recall', 'f1_score']].add_prefix('pooled_'))
print('Equal-weight monthly mean ± std, and pooled over all four months '
      '(tuned models use their CV threshold, defaults use 0.5):')
display(summary.round(3))
display(pooled.round(3))

# Save tuned parameters and thresholds

In [ ]:
os.makedirs('results', exist_ok=True)
tuning_output = {
    'run_date': RUN_DATE, 'cv': f'{N_CV_FOLDS}-fold day-blocked expanding window',
    'scoring': 'average_precision',
    'models': {name: {'family': MODEL_FAMILY[name], 'best_params': params_to_json(best_params[name]),
                      'f1_threshold': thresholds[name],
                      'cv_pr_auc_default': cv_results.loc[(name, 'default'), 'cv_pr_auc_mean'],
                      'cv_pr_auc_tuned': cv_results.loc[(name, 'tuned'), 'cv_pr_auc_mean']}
               for name in tuned_models},
}
with open('results/classification_tuning.json', 'w') as f:
    json.dump(tuning_output, f, indent=2)
print(json.dumps(tuning_output, indent=2)[:1500])

# Using the results in `model_classification_dev.ipynb`
```python
import json
tuning = json.load(open('results/classification_tuning.json'))
lgb_pipeline.set_params(**tuning['models']['LightGBM']['best_params'])
PREDICTION_THRESHOLD = tuning['models']['LightGBM']['f1_threshold']
```
The parameter names (`classifier__...`) match the dev notebook's pipeline step names.

# Notes for the report
- **Model family budget.** The brief allows 2–3 families. Here, Logistic Regression is the *Linear* family, and Decision Tree, Random Forest, XGBoost and LightGBM are all *Tree-based*. Each family starts from its simplest variant (plain LR; a single decision tree). XGBoost and LightGBM are close alternatives, so the report could keep one and put the other in the appendix.
- **Tuning method.** Random search over small grids on 5 expanding-window folds, scored by PR-AUC. The threshold comes from out-of-fold F1. Seeds are fixed (`RANDOM_STATE = 42`), and all preprocessing is fitted inside each pipeline and fold.
- **Evaluation.** Report CV mean ± std, hold-out metrics, and the April–July backtest (monthly mean ± std and pooled). The backtest months come after all data used for tuning.
- **Limitations.**
  - Grids are small; widen them once the features are frozen.
  - The threshold is tuned for F1, not for a business cost or capacity.
  - Scores are not calibrated probabilities.